# Semantic MentalRoBERTa Embedding Extraction

This notebook extracts one 768-dimensional CLS embedding per sample/post using MentalRoBERTa. It is intended for Google Colab GPU runtime.

Outputs are split-aware parquet files:

- `data/features/semantic/mental_roberta_train.parquet`
- `data/features/semantic/mental_roberta_val.parquet`
- `data/features/semantic/mental_roberta_test.parquet`

Each parquet has columns: `post_id`, `features`.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip -q install transformers accelerate pandas pyarrow tqdm huggingface_hub

## 3. Optional Hugging Face login

Public models usually do not need login. If the model download fails because access is gated, run this cell and paste your Hugging Face token.

In [ ]:
# Optional: uncomment if needed
# from huggingface_hub import login
# login()

## 4. Mount Google Drive and set paths

Upload/copy your project folder to Drive, then set `PROJECT_DIR` to that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# CHANGE THIS if your project is stored elsewhere in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/mental_health_fusion')

DATA_DIR = PROJECT_DIR / 'data' / 'processed'
OUT_DIR = PROJECT_DIR / 'data' / 'features' / 'semantic'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)

## 5. Configure extraction

Expected input CSVs:

- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`

The text column should be `text`. If there is no `post_id` column, the notebook creates stable IDs from the split name and row index.

In [ ]:
MODEL_NAME = 'mental/mental-roberta-base'
TEXT_COL = 'text'
POST_ID_COL = 'post_id'
MAX_LENGTH = 512
BATCH_SIZE = 32  # lower to 16 or 8 if Colab runs out of memory

SPLITS = {
    'train': DATA_DIR / 'train.csv',
    'val': DATA_DIR / 'val.csv',
    'test': DATA_DIR / 'test.csv',
}

for split, path in SPLITS.items():
    print(split, path, path.exists())

## 6. Load model

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print('loaded:', MODEL_NAME)

## 7. Extraction function

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE):
    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc='batches'):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        cls = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy().astype('float32')
        all_embeddings.append(cls)
    return np.vstack(all_embeddings)

def extract_split(split, csv_path, force=False):
    out_path = OUT_DIR / f'mental_roberta_{split}.parquet'
    if out_path.exists() and not force:
        print(f'Skipping {split}: exists -> {out_path}')
        return out_path

    df = pd.read_csv(csv_path)
    if TEXT_COL not in df.columns:
        raise KeyError(f'Missing text column {TEXT_COL} in {csv_path}')

    if POST_ID_COL in df.columns:
        post_ids = df[POST_ID_COL].astype(str).tolist()
    else:
        post_ids = [f'{split}_{i}' for i in range(len(df))]

    texts = df[TEXT_COL].fillna('').astype(str).tolist()
    embeddings = encode_texts(texts)
    assert embeddings.shape == (len(df), 768), embeddings.shape

    out_df = pd.DataFrame({
        'post_id': post_ids,
        'features': embeddings.tolist(),
    })
    out_df.to_parquet(out_path, index=False)
    print(f'Saved {split}: {out_path} shape={embeddings.shape}')
    return out_path

## 8. Run extraction

Start with `val` or `test` first to verify everything works. Then run `train`.

In [ ]:
# Quick check first
extract_split('val', SPLITS['val'], force=False)

In [ ]:
# Then run the full train split
extract_split('train', SPLITS['train'], force=False)

In [ ]:
# Finally test split
extract_split('test', SPLITS['test'], force=False)

## 9. Verify output

In [ ]:
for split in ['train', 'val', 'test']:
    path = OUT_DIR / f'mental_roberta_{split}.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        print(split, len(df), len(df.iloc[0]['features']))